# 02 · Spatial Transformer Network Module (Hannah)

A Spatial Transformer Network (Jaderberg et al., 2015) has three parts:

1. **Localization network** – a small CNN that looks at the input image and predicts affine parameters θ (2×3 matrix).
2. **Grid generator** – builds a sampling grid from θ.
3. **Sampler** – bilinearly resamples the input on that grid.

The result is an image that is re-scaled / rotated / sheared before it reaches the classifier, which helps with the scale, rotation, and viewing-angle changes in traffic signs.

The last localization layer is initialised to the **identity transform**, so training starts from "do nothing" and the STN only learns to warp when it helps.

This notebook writes `src/models/stn.py`, which contains:
- `STN` – the module itself
- `STNClassifier` – wraps *any* classifier: `x → STN → classifier`
- `visualize_stn` – shows inputs vs. STN outputs

In [ ]:
# --- Locate the project root so `src/` imports work from any folder ---
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
%%writefile src/models/stn.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class STN(nn.Module):
    """Affine Spatial Transformer. Input and output shape: (B, C, H, W)."""

    def __init__(self, in_channels: int = 3, img_size: int = 48):
        super().__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=7),
            nn.MaxPool2d(2),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 10, kernel_size=5),
            nn.MaxPool2d(2),
            nn.ReLU(inplace=True),
        )
        # Work out the flattened size for whatever img_size is used
        with torch.no_grad():
            n_feat = self.localization(torch.zeros(1, in_channels, img_size, img_size)).numel()

        self.fc_loc = nn.Sequential(
            nn.Linear(n_feat, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 6),
        )
        # Start as the identity transform: [[1, 0, 0], [0, 1, 0]]
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

        self.last_theta = None   # stored for inspection / visualisation

    def forward(self, x, return_theta: bool = False):
        theta = self.fc_loc(self.localization(x).flatten(1)).view(-1, 2, 3)
        self.last_theta = theta.detach()

        # Do the sampling in float32 even under mixed precision (grid_sample is precision-sensitive)
        with torch.autocast(device_type=x.device.type, enabled=False):
            theta32, x32 = theta.float(), x.float()
            grid = F.affine_grid(theta32, x32.size(), align_corners=False)
            out = F.grid_sample(x32, grid, mode="bilinear",
                                padding_mode="border", align_corners=False)
        out = out.to(x.dtype)
        return (out, theta) if return_theta else out


class STNClassifier(nn.Module):
    """x -> STN -> classifier. `classifier` can be any nn.Module returning logits."""

    def __init__(self, stn: nn.Module, classifier: nn.Module):
        super().__init__()
        self.stn = stn
        self.classifier = classifier

    def forward(self, x):
        return self.classifier(self.stn(x))


@torch.no_grad()
def visualize_stn(model, loader, device, n: int = 8, save_path=None):
    """Top row: original images. Bottom row: after the STN.
    Assumes images are in [0, 1] (see the interface notes in notebook 00)."""
    import matplotlib.pyplot as plt

    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    n = x.size(0)
    x_t = model.stn(x)

    fig, axes = plt.subplots(2, n, figsize=(1.6 * n, 3.6), squeeze=False)
    for i in range(n):
        axes[0, i].imshow(x[i].cpu().permute(1, 2, 0).clamp(0, 1)); axes[0, i].axis("off")
        axes[1, i].imshow(x_t[i].cpu().permute(1, 2, 0).clamp(0, 1)); axes[1, i].axis("off")
    axes[0, 0].set_title("input", fontsize=9, loc="left")
    axes[1, 0].set_title("after STN", fontsize=9, loc="left")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    plt.show()

## Sanity checks

In [ ]:
import torch, torch.nn as nn
from src.models.stn import STN, STNClassifier

stn = STN(in_channels=3, img_size=48)
x = torch.rand(4, 3, 48, 48)

# 1) Shape preserved
y, theta = stn(x, return_theta=True)
assert y.shape == x.shape
print("output shape:", tuple(y.shape), "| theta shape:", tuple(theta.shape))

# 2) Identity at initialisation -> output equals input
identity = torch.tensor([[1., 0., 0.], [0., 1., 0.]]).expand(4, 2, 3)
assert torch.allclose(theta, identity), "theta should start as identity"
print("identity init OK, max |out - in| =", (y - x).abs().max().item())

# 3) Gradients reach the localization network
y.mean().backward()
assert stn.fc_loc[2].bias.grad is not None
print("gradients flow to localization net: OK")

# 4) Works inside a classifier wrapper (trivial placeholder classifier, just for the test)
dummy_clf = nn.Sequential(nn.Flatten(), nn.Linear(3 * 48 * 48, 43))
wrapped = STNClassifier(STN(3, 48), dummy_clf)
print("wrapped output:", tuple(wrapped(x).shape))

# 5) Size of the STN itself
print("STN parameters:", sum(p.numel() for p in stn.parameters()))

**Next steps:** notebooks `04` and `05` plug this module in front of Alex's CNN models. Because `STNClassifier` accepts any classifier, the ablation variants all share the same STN code.